In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
import os

# Uyarıları kapatalım (Opsiyonel)
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported.")

✅ Libraries imported.


In [2]:
# --- 1. DATA LOADING ---
df = pd.read_csv("../02_Processed_Data/06_Training_Data.csv")

# --- 2. DEFINE FEATURES AND TARGET ---
y = df['chatter']


feature_columns = [
    
    'IMotor_std',
    'Z_Accel_std',
    'VibRes_Accel_std',
    'Trq_std',
    
    'Z_HighFreq_RMS',
    'IMotor_HighFreq_RMS',
    'Trq_HighFreq_RMS',

    'IMotor_std_Ratio_to_Past',
    'Z_Accel_std_Ratio_to_Past',
    'VibRes_Accel_std_Ratio_to_Past',
    'Trq_std_Ratio_to_Past',
    'IMotor_HighFreq_RMS_Ratio_to_Past',
    'Z_HighFreq_RMS_Ratio_to_Past',

    'IMotor_std_Trend',
    'Z_Accel_std_Trend',
    'VibRes_Accel_std_Trend',
    'Trq_std_Trend',
    'IMotor_HighFreq_RMS_Trend',
    'Z_HighFreq_RMS_Trend'
]

# Girdi matrisi (X)
X = df[feature_columns]

print(f"✅ Data loaded: {X.shape[0]} rows and {X.shape[1]} features.")

✅ Data loaded: 25307 rows and 19 features.


In [3]:
# --- 1. CALCULATE CLASS WEIGHTS ---
# Calculating scale_pos_weight for class imbalance
count_0 = (y == 0).sum()
count_1 = (y == 1).sum()
weight_for_1 = (count_0 / count_1 ) * 0.7

print(f"Number of Class 0: {count_0} | Number of Class 1: {count_1}")
print(f"Calculated scale_pos_weight: {weight_for_1:.2f}\n")

# --- 2. FINAL MODEL SETUP ---
# Set the optimal number of trees calculated from the LOOCV average (303 trees)
optimal_n_estimators = 303 

final_model = XGBClassifier(
    n_estimators=optimal_n_estimators, # The calculated average from 20 folds
    learning_rate=0.05,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    scale_pos_weight=weight_for_1,
    random_state=42,
    reg_lambda=2.0,
    reg_alpha=0.5
    
)

print(f"✅ Final Model structured with {optimal_n_estimators} trees. Ready for training.")

Number of Class 0: 17647 | Number of Class 1: 7660
Calculated scale_pos_weight: 1.61

✅ Final Model structured with 303 trees. Ready for training.


In [4]:
# --- 1. FIT THE FINAL MODEL ---
print(f"🚀 Training Final Model with {optimal_n_estimators} trees on 100% of the data...")

# Fit the model on the entire dataset (X and y)
final_model.fit(X, y)

print("✅ Final Model training completed!")

🚀 Training Final Model with 303 trees on 100% of the data...
✅ Final Model training completed!


In [5]:
import os

# --- 1. EXPORT THE MODEL FOR SHAP ANALYSIS ---
# Create the requested models directory if it doesn't exist
models_dir = "../03_Models"
os.makedirs(models_dir, exist_ok=True)

# Define the file path with the specified name
model_path = os.path.join(models_dir, "01_XGB_Model.json")

# Save the trained final model
final_model.save_model(model_path)

print(f"💾 Final Model saved successfully to:\n📂 {model_path}")

💾 Final Model saved successfully to:
📂 ../03_Models/01_XGB_Model.json
